In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import pipeline
from transformers.utils import logging
import warnings
import gradio as gr
import torch

# 1. Load model and tokenizer
model_name = "distilgpt2"
print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
print("Model loaded successfully!")

# 2. Define prediction function with fixed string decoding
def predict(message, history):
    try:
        # Build conversation context safely from history
        context = ""
        if history:
            for turn in history:
                if isinstance(turn, (list, tuple)) and len(turn) == 2:
                    user_msg, bot_msg = turn
                    if user_msg:
                        context += f"{user_msg}{tokenizer.eos_token}"
                    if bot_msg:
                        context += f"{bot_msg}{tokenizer.eos_token}"

        # Add current message
        context += f"{message}{tokenizer.eos_token}"

        # Encode inputs
        input_ids = tokenizer.encode(context, return_tensors='pt')

        # Generate response
        output_ids = model.generate(
            input_ids,
            max_length=input_ids.shape[-1] + 100,
            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=3,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7
        )

        # Decode only the newly generated tokens
        response_ids = output_ids[:, input_ids.shape[-1]:]

        # Ensure we decode properly into a single string
        bot_response = tokenizer.decode(response_ids[0], skip_special_tokens=True)

        return bot_response.strip() if bot_response.strip() else "Hmm, I didn't catch that. Try again!"

    except Exception as e:
        return f"An error occurred: {str(e)}"

# 3. Launch Gradio Chat UI
demo = gr.ChatInterface(
    fn=predict,
    title="Transformer Chatbot (DistilGPT-2)",
    description="A simple chatbot powered by a Hugging Face transformer model running live via Gradio."
)

if __name__ == "__main__":
    demo.launch(share=True, debug=True)

Loading model and tokenizer...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d7e5a096b309244b69.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
